# SB100 Squad 2 — extrair um lote baixado do site

Este notebook **não precisa de chave do banco**. O caminho é:

1. No site, aba **Extração**, baixe um lote (`lote_....zip`): PDFs ainda não
   extraídos, com os metadados da curadoria.
2. Aqui, envie esse zip e rode as células.
3. Baixe o `extracao_....zip` que sai no fim.
4. No site, aba **Extração**, envie esse zip. O site confere e grava.

Roda no Colab (escolha **GPU** em Ambiente de execução → Alterar tipo) ou num
Jupyter de servidor com placa.

Se a sessão cair, rode de novo com o mesmo lote: o que já terminou é pulado.


## 1. Preparo


In [ ]:
import os, sys, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "SEM GPU - no Colab, troque em Ambiente de execucao")

DENTRO_DO_COLAB = "google.colab" in sys.modules
BASE = "/content" if DENTRO_DO_COLAB else os.path.abspath("trabalho")
os.makedirs(BASE, exist_ok=True)
os.environ["SB100_DIR"] = BASE

%cd {BASE}
!rm -rf {BASE}/repo {BASE}/squad-2
# o repositório da organização tem outros projetos: baixa só a pasta do site (~1 MB)
!git clone -q --depth 1 --filter=blob:none --sparse https://github.com/CCDSB100/squad-2.git {BASE}/squad-2
!git -C {BASE}/squad-2 sparse-checkout set 2026/parser_rag_framework
!mv {BASE}/squad-2/2026/parser_rag_framework {BASE}/repo && rm -rf {BASE}/squad-2
print("pasta de trabalho:", BASE)


## 2. Enviar o lote

No Colab abre um seletor: escolha o `lote_....zip` baixado do site.
Fora do Colab, preencha `CAMINHO_DO_LOTE`.


In [ ]:
CAMINHO_DO_LOTE = ""  #@param {type:"string"}

if DENTRO_DO_COLAB and not CAMINHO_DO_LOTE:
    from google.colab import files
    enviados = files.upload()
    CAMINHO_DO_LOTE = os.path.join(BASE, next(iter(enviados)))
assert CAMINHO_DO_LOTE.lower().endswith(".zip"), "envie o lote_....zip baixado do site"
assert os.path.exists(CAMINHO_DO_LOTE), f"nao achei {CAMINHO_DO_LOTE}"
print("lote:", CAMINHO_DO_LOTE)


## 3. Extrair

**HIBRIDO**: `False` passa toda página pelo Chandra (como o Boletim 100);
`True` usa o Docling nas páginas de texto sem tabela.

**CHANDRA_MAX_LADO**: lado maior da imagem entregue ao modelo. 2200 lê
melhor rótulo de eixo e expoente; o custo cresce com a área.

**LOTE_PAGINAS**: páginas por chamada. Numa L4 a GPU ficou em 51% com 1,
então 4 tende a render. Se estourar memória, volte para 1.

Na primeira vez baixa o modelo do Chandra (10,6 GB, ~1 min).


In [ ]:
HIBRIDO = False          #@param {type:"boolean"}
CHANDRA_MAX_LADO = 2200  #@param {type:"integer"}
LOTE_PAGINAS = 1         #@param {type:"integer"}

os.environ["CHANDRA_MAX_LADO"] = str(CHANDRA_MAX_LADO)
os.environ["LOTE_PAGINAS"] = str(LOTE_PAGINAS)
extra = "--hibrido" if HIBRIDO else ""
!python {BASE}/repo/extractor/rodar_lote.py {CAMINHO_DO_LOTE} {extra}


## 4. Baixar o resultado

Este é o zip para enviar de volta na aba **Extração** do site.


In [ ]:
import glob
feitos = sorted(glob.glob(os.path.join(BASE, "extracao_*.zip")), key=os.path.getmtime)
assert feitos, "nenhum extracao_*.zip: veja a saida da celula 3"
RESULTADO = feitos[-1]
print("resultado:", RESULTADO, f"({os.path.getsize(RESULTADO)/2**20:.1f} MB)")
if DENTRO_DO_COLAB:
    from google.colab import files
    files.download(RESULTADO)
